<a href="https://colab.research.google.com/github/LaraDondossola/Classificacao-eventos-climaticos/blob/main/Notebooks/Interface_eventos_clim%C3%A1ticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit joblib pandas scikit-learn
!npm install -g localtunnel
!pip install -q pyngrok streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 94.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# --- CONFIGURAÇÃO DA PÁGINA ---
st.set_page_config(
    page_title="Predição de Eventos Climáticos",
    page_icon="⛈️",
    layout="wide"
)

# --- CSS PARA OCULTAR OS BOTÕES DE INCREMENTO/DECREMENTO ---
st.markdown("""
    <style>
    button[data-testid="stNumberInputStepDown"],
    button[data-testid="stNumberInputStepUp"] {
        display: none !important;
    }
    input[type=number]::-webkit-inner-spin-button,
    input[type=number]::-webkit-outer-spin-button {
        -webkit-appearance: none;
        margin: 0;
    }
    input[type=number] {
        -moz-appearance: textfield;
    }
    </style>
""", unsafe_allow_html=True)

# --- CORTES DE RISCO ---
# valores impressos na célula de treino (corte_medio e corte_alto)
CORTE_BAIXO_MEDIO = 3.6034   # <-- valor de corte_medio (mediana, em %)
CORTE_MEDIO_ALTO = 50.1603    # <-- valor de corte_alto (P90, em %)

def classificar_risco_regra(populacao_total, populacao_afetada):
    if populacao_total <= 0:
        return "Indefinido"
    percentual = (populacao_afetada / populacao_total) * 100
    if percentual >= CORTE_MEDIO_ALTO:
        return "Alto"
    elif percentual >= CORTE_BAIXO_MEDIO:
        return "Médio"
    else:
        return "Baixo"

# Título e Descrição
st.title("⛈️ Painel de Avaliação de Impacto e Risco Climático")
st.markdown("Insira os dados da ocorrência abaixo para estimar a **População Afetada** e calcular o **Nível de Risco** (regra sobre o percentual estimado de população afetada).")

# --- CARREGAMENTO DO MODELO E PREPROCESSOR ---
@st.cache_resource
def carregar_artefatos():
    base_path = Path("Models") if Path("Models").exists() else Path(".")
    preprocessor = joblib.load(base_path / "preprocessor.pkl")
    modelo_reg = joblib.load(base_path / "modelo_1_regressao_knn.pkl")
    return preprocessor, modelo_reg

try:
    preprocessor, modelo_reg = carregar_artefatos()
    st.sidebar.success("✅ Modelo de Regressão e Preprocessor carregados com sucesso!")
except Exception as e:
    st.error(f"Erro ao carregar o modelo: {e}")
    st.stop()

# --- FORMULÁRIO DE ENTRADA DE DADOS ---
st.header("📋 Dados da Ocorrência")

meses_map = {
    "Janeiro": 1, "Fevereiro": 2, "Março": 3, "Abril": 4,
    "Maio": 5, "Junho": 6, "Julho": 7, "Agosto": 8,
    "Setembro": 9, "Outubro": 10, "Novembro": 11, "Dezembro": 12
}

col1, col2, col3 = st.columns(3)

with col1:
    uf = st.selectbox("Estado (UF)", options=[
        'AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA',
        'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN',
        'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO'
    ], index=23)

    populacao = st.number_input("População Total do Município", min_value=0, value=25000)

with col2:
    tipo_evento = st.selectbox("Tipo de Evento", options=[
        'Enxurradas', 'Inundações', 'Alagamentos', 'Tempestade Local/Convectiva',
        'Inundação', 'Vendaval / Ciclone', 'Estiagem', 'Seca', 'Granizo',
        'Geada', 'Incêndio Florestal', 'Deslizamentos', 'Outros'
    ], index=0)

    nome_mes = st.selectbox("Mês do Registro", options=list(meses_map.keys()), index=5)
    mes = meses_map[nome_mes]

    trimestre = (mes - 1) // 3 + 1
    st.info(f"🗓️ Trimestre estimado: **{trimestre}º Trimestre**")

with col3:
    hab_danificadas = st.number_input("Habitações Danificadas", min_value=0, value=0)
    hab_destruidas = st.number_input("Habitações Destruídas", min_value=0, value=0)
    infra_danificada = st.number_input("Obras de Infraestrutura Pública Danificadas", min_value=0, value=0)

# --- BOTÃO DE PREDIÇÃO E PROCESSAMENTO ---
st.markdown("---")

if st.button("🚀 Calcular Previsões", type="primary", use_container_width=True):
    try:
        try:
            colunas_esperadas = preprocessor.feature_names_in_
        except AttributeError:
            colunas_esperadas = preprocessor.steps[0][1].feature_names_in_

        col_categoricas = []
        col_numericas = []

        if hasattr(preprocessor, 'transformers_'):
            for name, trans, cols in preprocessor.transformers_:
                if name != 'remainder':
                    if any(c in str(trans).lower() for c in ['onehot', 'ordinal', 'encoder', 'categorical']):
                        col_categoricas.extend(cols)
                    else:
                        col_numericas.extend(cols)

        dados_dict = {}
        for col in colunas_esperadas:
            if col in col_categoricas or col in ['UF', 'Tipo_Evento']:
                dados_dict[col] = "Outros"
            else:
                dados_dict[col] = 0.0

        dados_dict.update({
            'UF': str(uf),
            'Tipo_Evento': str(tipo_evento),
            'População': float(populacao),
            'Mes_Registro': float(mes),
            'Trimestre': float(trimestre),
            'DM_Unidades Habitacionais Danificadas': float(hab_danificadas),
            'DM_Unidades Habitacionais Destruídas': float(hab_destruidas),
            'DM_Obras de infraestrutura pública Danificadas': float(infra_danificada)
        })

        dados_entrada = pd.DataFrame([dados_dict])[colunas_esperadas]

        for col in dados_entrada.columns:
            if col not in ['UF', 'Tipo_Evento'] and col not in col_categoricas:
                dados_entrada[col] = pd.to_numeric(dados_entrada[col], errors='coerce').fillna(0.0)

        dados_processados = preprocessor.transform(dados_entrada)

        # Predição da População Afetada (Regressão)
        pred_pop_afetada_raw = modelo_reg.predict(dados_processados)[0]
        pred_pop_afetada = max(0, int(round(pred_pop_afetada_raw)))

        # Nível de Risco calculado por regra (percentual estimado de população afetada)
        pred_risco = classificar_risco_regra(populacao, pred_pop_afetada)

        pop_formatada = f"{pred_pop_afetada:,}".replace(",", ".")

        res_col1, res_col2 = st.columns(2)

        with res_col1:
            st.subheader("🔴 Nível de Risco Estimado")
            st.metric(label="Risco (por regra)", value=pred_risco)

        with res_col2:
            st.subheader("👥 População Afetada Estimada")
            st.metric(label="Total de Pessoas Afetadas", value=f"{pop_formatada} pessoas")

    except Exception as err:
        st.error(f"Erro ao processar as previsões: {err}")

Overwriting app.py


In [1]:
from pyngrok import ngrok
import os

# 1. Configurar o seu Authtoken (substitua com o token do site)
ngrok.set_auth_token("xx")

# 2. Encerrar conexões anteriores
ngrok.kill()

# 3. Criar o túnel na porta 8501
public_url = ngrok.connect(8501)
print(f"🔗 Acesse sua aplicação sem erros aqui: {public_url}")

# 4. Rodar o Streamlit
!streamlit run app.py --server.port 8501

ModuleNotFoundError: No module named 'pyngrok'